In [39]:
# setup

import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()
con.sql("CREATE VIEW business AS SELECT * FROM '../data/processed/business.parquet'")
con.sql("CREATE VIEW review AS SELECT * FROM '../data/processed/review.parquet'")

# business as a pd dataframe (use SQL and pandas to profile! yay)
business = con.sql("SELECT * FROM business").df()

In [40]:
# looking at columns

print(business.columns)

columns_r = con.sql('SELECT * FROM review').columns
print(columns_r)

Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'attributes', 'categories', 'hours'],
      dtype='str')
['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']


In [71]:
# how many reviews are there, and in what date range?

# review count
r_count = con.sql('SELECT COUNT(*) FROM review').fetchone()[0]
print(f'Review count: {r_count:,}')

# date range
r_dmin, r_dmax = con.sql('SELECT MIN(date), MAX(date) FROM review').fetchone()
print(f'date dtype: {type(r_dmin)}')
print(f'MIN date: {r_dmin}')
print(f'MAX date: {r_dmax}')

Review count: 6,990,280
date dtype: <class 'datetime.datetime'>
MIN date: 2005-02-16 03:23:22
MAX date: 2022-01-19 19:48:45


__1. reviews and date range__
1. Reviews: 6,990,280 total
2. Date range: Feb 2005 - Jan 2022

Date range includes COVID data. Dates stored as datetime.datetime

In [75]:
# how many restaurants? how many businesses have no categories?

# restaurant count
b_rest = business[business['categories'].str.contains('Restaurants', na=False)]
print(f'Restaurant count: {b_rest.shape[0]:,}')

# businesses tagged "Food" but not "Restaurants"
cats = business["categories"]
has_food = cats.str.contains("Food", na=False)
has_rest = cats.str.contains("Restaurants", na=False)

food_only = business[has_food & ~has_rest]
print(f"\nNumber of businesses marked as Food but not Restaurants: {len(food_only)}")

# most common categories among them
top_cats = (
    food_only["categories"]
    .str.split(", ")
    .explode()
    .value_counts()
    .drop("Food", errors="ignore")
    .head(15)
)
print(top_cats)

# counting empty categories 
text_cols = ["name", "address", "city", "state", "postal_code", "categories"]
b = business[text_cols]

missing = pd.DataFrame({
    "nulls": b.isna().sum(),
    "empty": (b == "").sum(),
    "whitespace_only": b.apply(lambda s: ((s.str.strip() == "") & (s != "")).sum()),
})
print(f'\nMissing data counts by column: \n{missing}')

Restaurant count: 52,268

Number of businesses marked as Food but not Restaurants: 12348
categories
Shopping                     2750
Coffee & Tea                 2650
Grocery                      2312
Specialty Food               2222
Ice Cream & Frozen Yogurt    1569
Convenience Stores           1384
Desserts                     1348
Drugstores                   1319
Bakeries                     1261
Beer                         1259
Wine & Spirits               1259
Juice Bars & Smoothies        669
Donuts                        622
Nightlife                     603
Health & Medical              566
Name: count, dtype: int64

Missing data counts by column: 
             nulls  empty  whitespace_only
name             0      0                0
address          0   5127                0
city             0      0                0
state            0      0                0
postal_code      0     73                0
categories     103      0                0


__2. restaurant count, businesses with no categories__
1. Restaurants: 52,268 total. 

Analyzing restaurants only. Businesses with food-only categories (no restaurant) appear to be grocery, coffee shops and convenience stores, which don't fit questions about meal quality. 12,348 businesses are tagged Food but not Restaurants. Top categories are Shopping (2,750), Coffee & Tea (2,650), Grocery (2,312), etc.

2. 5,127 empty values under address, 73 under postal_code, and 103 nulls for categories.

Missing values are stored inconsistently: empty strings in address and postal code, nulls in categories. A null-only check would report address as complete. 

Standardize empty strings to null in src/clean.py.

In [76]:
# how many reviews belong to restaurants?

rb_count = con.sql("""SELECT COUNT (*) FROM review r 
                    JOIN business b ON r.business_id = b.business_id
                    WHERE b.categories LIKE '%Restaurants%'""").fetchone()[0]

print(f'Restaurant review count: {rb_count:,}')
print(f'% of total review count: {rb_count/r_count :.2%}')

Restaurant review count: 4,724,471
% of total review count: 67.59%


__3. reviews belonging to restaurants__
1. Yelp dataset has 4,724,471 reviews belonging to restaurants, which is about 67.6% of all reviews.